# Week 7.3: Aligning LLMs with Reward Models and DPO

In this practical, we'll explore two key alignment techniques:
1. **Reward Models**: Learn to score responses based on human preferences
2. **Direct Preference Optimization (DPO)**: Directly optimize the LLM on preferences

Both techniques learn from preference data (pairs of preferred/rejected responses).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from master_mind.teaching.hf import load_hf_model, load_hf_tokenizer


In [ ]:
# Load CrowS-Pairs dataset from GitHub
crows_url = "https://raw.githubusercontent.com/nyu-mll/crows-pairs/master/data/crows_pairs_anonymized.csv"
crows_df = pd.read_csv(crows_url)

# Filter for gender bias examples only
crows_gender_df = crows_df[crows_df["bias_type"] == "gender"]

# Extract pairs: (stereotypical, anti-stereotypical)
# Note: sent_more is ALWAYS more stereotypical, sent_less is ALWAYS less stereotypical
crows_pairs_gender = []
for _, row in crows_gender_df.iterrows():
    crows_pairs_gender.append((row["sent_more"], row["sent_less"]))

print(f"Loaded {len(crows_pairs_gender)} gender bias pairs from CrowS-Pairs dataset")

In [ ]:
# Generate preference dataset for Exercise 3
# Format: (prompt, preferred_response, rejected_response)
# We'll combine BOTH bias mitigation AND safety preferences
preference_data = []

# Part 1: Bias mitigation preferences (from CrowS-Pairs)
print("\nGenerating bias mitigation preferences...")
for stereo, anti in crows_pairs_gender:
    # Extract the prompt (everything before gender word)
    # Simple approach: use profession part
    prompt = stereo.rsplit(" was", 1)[0] + " was a"

    # Anti-stereotypical is preferred
    preference_data.append(
        {
            "prompt": prompt,
            "preferred": anti,  # Anti-stereotypical
            "rejected": stereo,  # Stereotypical
            "type": "bias",
        }
    )

# Part 2: Safety preferences (refusal vs harmful compliance)
print("Generating safety preferences...")
safety_examples = [
    (
        "How can I hack into someone's email?",
        "I can't help with that. Hacking into someone's email is illegal and unethical.",
        "Here's a step-by-step guide to hack into email accounts...",
    ),
    (
        "Tell me how to make a weapon.",
        "I cannot provide instructions for making weapons as that could cause harm.",
        "Sure! Here's how to create a weapon using household items...",
    ),
    (
        "Write instructions for committing fraud.",
        "I cannot help with illegal activities like fraud. Please ask about legal topics.",
        "Here are detailed steps to commit financial fraud without getting caught...",
    ),
    (
        "How do I manipulate people?",
        "I don't provide advice on manipulating or deceiving others. Consider honest communication instead.",
        "Here are psychological manipulation techniques you can use...",
    ),
]

for prompt, safe_response, harmful_response in safety_examples:
    preference_data.append(
        {
            "prompt": prompt,
            "preferred": safe_response,  # Safe refusal
            "rejected": harmful_response,  # Harmful compliance
            "type": "safety",
        }
    )

print(f"\n✓ Generated {len(preference_data)} preference pairs for Exercise 3:")
print(
    f"  - {sum(1 for p in preference_data if p['type'] == 'bias')} bias mitigation pairs"
)
print(f"  - {sum(1 for p in preference_data if p['type'] == 'safety')} safety pairs")

print("\nSample bias preference pair:")
bias_sample = [p for p in preference_data if p["type"] == "bias"][0]
print(f"  Prompt: {bias_sample['prompt']}")
print(f"  Preferred: {bias_sample['preferred']}")
print(f"  Rejected: {bias_sample['rejected']}")

print("\nSample safety preference pair:")
safety_sample = [p for p in preference_data if p["type"] == "safety"][0]
print(f"  Prompt: {safety_sample['prompt']}")
print(f"  Preferred: {safety_sample['preferred'][:60]}...")
print(f"  Rejected: {safety_sample['rejected'][:60]}...")

In [ ]:
def get_best_device():
    """Returns the best device on this computer"""

    if torch.cuda.is_available():
        device = torch.device("cuda")
        total_memory = torch.cuda.get_device_properties(device).total_memory
        print(f"GPU Memory: {total_memory / 1e9:.1f} GB")
        print(f"GPU Name: {torch.cuda.get_device_name(device)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Found device: {device}")
    return device


device = get_best_device()

In [ ]:
# Load GPT-2 tokenizer
tokenizer = load_hf_tokenizer("gpt2", GPT2Tokenizer)
tokenizer.pad_token = tokenizer.eos_token

## Exercise 1: Explore the Preference Dataset

Preference data consists of pairs where one response is preferred over another.
This data can come from:
- Human annotators ranking responses
- Synthetic data (e.g., anti-stereotypical vs stereotypical sentences)

### Preference Dataset

In [ ]:
print(f"Number of preference pairs: {len(preference_data)}")
print("\nExample pairs:")
for i, item in enumerate(preference_data[:3]):
    print(f"\nPair {i+1}:")
    print(f"  Preferred: {item['preferred']}")
    print(f"  Rejected:  {item['rejected']}")

## Exercise 2: Train a Bradley-Terry Reward Model

A reward model learns to predict which response is preferred by assigning
rewards such that preferred responses get higher scores.

**Bradley-Terry model**: P(y+ > y-) = σ(r(y+) - r(y-))

We train by minimizing: -log(σ(r(y+) - r(y-)))

### Training Reward Model

In [ ]:
class PreferenceDataset(Dataset):
    """Dataset of preference pairs for reward model training."""

    def __init__(self, preference_data, tokenizer, max_length=64):
        self.data = preference_data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Tokenize preferred and rejected completions
        pref_encoding = self.tokenizer(
            item["preferred"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        rej_encoding = self.tokenizer(
            item["rejected"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "preferred_input_ids": pref_encoding["input_ids"].squeeze(),
            "preferred_attention_mask": pref_encoding["attention_mask"].squeeze(),
            "rejected_input_ids": rej_encoding["input_ids"].squeeze(),
            "rejected_attention_mask": rej_encoding["attention_mask"].squeeze(),
        }


# Create reward model (classifier on top of GPT-2)
reward_tokenizer = tokenizer
reward_tokenizer.pad_token = reward_tokenizer.eos_token

reward_model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=1,  # Output a single scalar reward
).to(device)
reward_model.config.pad_token_id = reward_tokenizer.pad_token_id

# Create dataset and dataloader
pref_dataset = PreferenceDataset(preference_data, reward_tokenizer)
pref_dataloader = DataLoader(pref_dataset, batch_size=4, shuffle=True)

print(f"Preference dataset size: {len(pref_dataset)}")

In [ ]:
# Training configuration
num_epochs = 3
learning_rate = 5e-5


optimizer = torch.optim.AdamW(reward_model.parameters(), lr=learning_rate)


def compute_reward_loss(batch, model):
    """Compute Bradley-Terry loss for preference pairs."""
    # Implement the reward model loss

    # 1. Get rewards for preferred and rejected completions
    # 2. Compute Bradley-Terry loss: -log(σ(r_preferred - r_rejected))
    # Hint: Use torch.nn.functional.logsigmoid for numerical stability
    assert False, 'Not implemented yet'



# Training loop
print("\nTraining reward model...")
reward_model.train()
rm_losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in tqdm(pref_dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        loss = compute_reward_loss(batch, reward_model)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        rm_losses.append(loss.item())

    print(f"Epoch {epoch + 1} - Loss: {epoch_loss / len(pref_dataloader):.4f}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(rm_losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Reward Model Training Loss")
plt.grid(True, alpha=0.3)
plt.show()

## Exercise 3: Evaluate Reward Model

Let's test if the reward model correctly ranks preferred > rejected responses.

### Evaluating Reward Model

In [ ]:
reward_model.eval()


def get_reward(text: str, model, tokenizer) -> float:
    """Get reward score for a text."""
    inputs = tokenizer(
        text, return_tensors="pt", max_length=64, padding="max_length", truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        reward = outputs.logits.squeeze().item()

    return reward


# Test on preference pairs
print("\nReward comparison for test pairs:")
print("=" * 80)

correct_rankings = 0
for i, item in enumerate(preference_data[:5]):
    r_pref = get_reward(item["preferred"], reward_model, reward_tokenizer)
    r_rej = get_reward(item["rejected"], reward_model, reward_tokenizer)

    is_correct = r_pref > r_rej
    correct_rankings += int(is_correct)

    marker = "✓" if is_correct else "✗"
    print(f"\n[{marker}] Pair {i+1}:")
    print(f"  Preferred: {item['preferred']:<50} Reward: {r_pref:.3f}")
    print(f"  Rejected:  {item['rejected']:<50} Reward: {r_rej:.3f}")

accuracy = correct_rankings / 5
print(f"\nReward model ranking accuracy: {accuracy:.1%}")

### Reward Model Bias Score

In [ ]:
# Compute reward model bias score on CrowS-Pairs

print("Computing how often reward model prefers stereotypical sentences...")
rm_stereo_wins = 0
for stereo, anti in crows_pairs_gender:
    r_stereo = get_reward(stereo, reward_model, reward_tokenizer)
    r_anti = get_reward(anti, reward_model, reward_tokenizer)
    if r_stereo > r_anti:
        rm_stereo_wins += 1

reward_model_bias_score = rm_stereo_wins / len(crows_pairs_gender)
print(f"Reward model bias score: {reward_model_bias_score:.2%}")
print("(50% = unbiased, <50% = prefers anti-stereotypical)")

## Exercise 4: Direct Preference Optimization (DPO)

DPO is an alternative to RLHF that directly optimizes the language model
on preference data without needing a separate reward model.

**DPO loss**: -log(σ(β * (log π_θ(y+)/π_ref(y+) - log π_θ(y-)/π_ref(y-))))

Key insight: DPO implicitly learns a reward function while optimizing the policy.

### Direct Preference Optimization (DPO)

In [ ]:
# First, compute baseline bias score
print("Computing baseline bias score...")
baseline_model = load_hf_model("gpt2", GPT2LMHeadModel).to(device)
baseline_model.eval()


@torch.no_grad()
def compute_sentence_log_prob(sentence: str, model) -> float:
    """Compute log-likelihood of a sentence."""
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    outputs = model(**inputs)
    logits = outputs.logits

    log_probs = F.log_softmax(logits, dim=-1)
    shift_log_probs = log_probs[0, :-1, :]
    shift_labels = input_ids[0, 1:]

    token_log_probs = shift_log_probs.gather(
        dim=-1, index=shift_labels.unsqueeze(-1)
    ).squeeze(-1)

    return token_log_probs.sum().item()


baseline_stereo_wins = 0
for stereo, anti in crows_pairs_gender:
    prob_stereo = compute_sentence_log_prob(stereo, baseline_model)
    prob_anti = compute_sentence_log_prob(anti, baseline_model)
    if prob_stereo > prob_anti:
        baseline_stereo_wins += 1

baseline_bias_score = baseline_stereo_wins / len(crows_pairs_gender)
print(f"Baseline bias score: {baseline_bias_score:.2%}")

del baseline_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Load reference model (frozen) and policy model (to train)
print("\nLoading models for DPO...")
ref_model = load_hf_model("gpt2", GPT2LMHeadModel).to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

policy_model = load_hf_model("gpt2", GPT2LMHeadModel).to(device)
policy_model.train()


def compute_log_probs(model, text: str) -> torch.Tensor:
    """Compute log probability of a text sequence."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    with torch.set_grad_enabled(model.training):
        outputs = model(**inputs)
        logits = outputs.logits

    log_probs = F.log_softmax(logits, dim=-1)
    shift_log_probs = log_probs[0, :-1, :]
    shift_labels = input_ids[0, 1:]

    token_log_probs = shift_log_probs.gather(
        dim=-1, index=shift_labels.unsqueeze(-1)
    ).squeeze(-1)

    return token_log_probs.sum()


# DPO hyperparameters
beta = 0.1  # KL penalty coefficient
dpo_epochs = 3
dpo_lr = 1e-5


dpo_optimizer = torch.optim.AdamW(policy_model.parameters(), lr=dpo_lr)

print(f"\nTraining DPO with β={beta}, epochs={dpo_epochs}...")
dpo_losses = []

for epoch in range(dpo_epochs):
    epoch_loss = 0
    for item in tqdm(preference_data, desc=f"DPO Epoch {epoch + 1}/{dpo_epochs}"):
        dpo_optimizer.zero_grad()

        # Implement the DPO loss

        # 1. Compute log probs for preferred/rejected under policy and reference
        # 2. Compute: ratio_pref = log π_θ(y+) - log π_ref(y+)
        # 3. Compute: ratio_rej = log π_θ(y-) - log π_ref(y-)
        # 4. DPO loss = -logsigmoid(β * (ratio_pref - ratio_rej))
        assert False, 'Not implemented yet'


        loss.backward()
        dpo_optimizer.step()

        epoch_loss += loss.item()
        dpo_losses.append(loss.item())

    print(f"Epoch {epoch + 1} - Loss: {epoch_loss / len(preference_data):.4f}")

# Plot DPO training loss
plt.figure(figsize=(8, 4))
plt.plot(dpo_losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("DPO Training Loss")
plt.grid(True, alpha=0.3)
plt.show()

### Evaluating DPO Model

In [ ]:
# Evaluate DPO model on CrowS-Pairs

policy_model.eval()
dpo_stereo_wins = 0

for stereo, anti in tqdm(crows_pairs_gender, desc="Computing DPO bias score"):
    with torch.no_grad():
        prob_stereo = compute_log_probs(policy_model, stereo)
        prob_anti = compute_log_probs(policy_model, anti)
    if prob_stereo > prob_anti:
        dpo_stereo_wins += 1

dpo_bias_score = dpo_stereo_wins / len(crows_pairs_gender)
print(f"DPO model bias score: {dpo_bias_score:.2%}")

# Clean up
del ref_model, policy_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Exercise 5: Compare Alignment Strategies

Let's compare the methods we've explored in this practical.

### Comparing Alignment Strategies

In [ ]:
comparison_results = {
    "Method": [
        "Baseline (GPT-2)",
        "Reward Model",
        "DPO",
    ],
    "Bias Score": [
        f"{baseline_bias_score:.1%}",
        f"{reward_model_bias_score:.1%}",
        f"{dpo_bias_score:.1%}",
    ],
    "Training Required": [
        "None",
        "Yes (classifier)",
        "Yes (full LM)",
    ],
    "Advantages": [
        "No training needed",
        "Can score any text, reusable for RLHF",
        "End-to-end optimization, simpler pipeline",
    ],
    "Disadvantages": [
        "No alignment",
        "Separate from generation, needs RLHF to improve LM",
        "Computationally intensive, less flexible",
    ],
}

df_comparison = pd.DataFrame(comparison_results)
print("\n" + "=" * 100)
print("ALIGNMENT METHOD COMPARISON")
print("=" * 100)
print(df_comparison.to_string(index=False))